Ordered Resolution converts clauses to _terminating_ Prolog programs



You should also table because there might be a lot of shared substructure. But you don't need it for termination


Ordered Rewriting become ground horn queries? What do do about unorderable clauses?
The body needs to be smaller?

I'm saying hereditary multset rewriting is senaible
But what about hereditary set "rewriting"
Finite generated equivalence classes of hereditary sets
finitely generated equivalence classes of hereditary multisets



https://www.metalevel.at/prolog/termination prolog termination
cti gupu
That new analyzer?
Ciao has stuff?

Is it "find a term order that makes all heads larger than bodies?"

ciaopp
https://ciao-lang.org/ciao/build/doc/ciaopp_tutorials.html/
https://arxiv.org/abs/2601.03849  Automated Theorem Proving for Prolog Verification
https://github.com/FredMesnard/LPTP


Ordered resolution finds a minimal fact database that would cause the given fact or set of facts to appear.
So yes, we do delete when rewriting.
It is adding an objective function sort of
out of all models I prefer the ones with p true. To tie break I prefer the ones with q
[p,q,r,...] give me the lexicographically minimal model.
You kind of _have_ to branch in the order of the variables?
But learned clauses may still prune the space.

critical pairs
p :- b1
p :- b2

means b1 = b2 (b1 :- b2 or b2 :- b1)

ground knuth bendix gets aways with murder.

It looks a bit more like ground multiset completion


mobius inversion
boolean function analysis. Fourier transform. Hmm.


The set of atoms is kind of the goal stack


Turn clauses into a prolog program st running the program finds a least model with the goal being true in it.
Convert clauses into something where ordered prolog works
Ordered prolog doesn't maintain a goal _stack_it retains a goal _maxheap_
You can probably implement this in a fun way


So this thing is a union find kind of.


Hmm. Could I go harrop with it? Struth talks of lattices but what about heyting word problems?
(a <= b)
Ground / propositional, so no exsitential or foralls. But 

```
a b c d
-------
   p
```

But then we aren't really free to swap around. The rewrite logic to the original's equational logic

What about the struth lattice interpretation

Ground prolog doesn't need backtracking really with the ordering.

Online good order finding. DPLL heuristics and the cdcl trail are a discovered ordering.

PTTP has all orderings of clauses but depth limits.


           | Equational | Propositional                                      | conditional equations        
unoriented |  equations | clauses p \/ ~q \/ r                               | 
orient     | rewrite    | rules   p :- q, not r  ===  (q /\ not r) => p      |

unoriented -> oriented  = knuth bendix
clauses -> rules  =  ordered resolution

a => b  ===  not a \/ b
binary oreintation ---> n-ary orientation


atomic equations <-> union find <-> ground atomi knuth bendix completion

propiositional clauses <-> ??? <-> ground ordered resoltuoin

a \/ ~b     b \/ c   a is maximal in clause 
------------------
   a \/ c


fourier motzkin 
a . x <= b . x

ax + by <= cx + dy

assymmetric completion
<=


eq(X,X) -> true
eq(t1,t2) = false

false = true



Rudi is right probably
If it should look liek assymmetric completion, some search is necessary
prolog should need to do backtracking
S and R relations?
So the "reduction" would be a set of things that might produce
Or the minimal set? but

I had no choice points in my prolog. Goals can be 
[{goal1, goal2},  {goal3, goal4}]


Proofs as sequsences of clauses such that it either follows fromr somrthing previous
[c1,c2,c3]

Linear resolution proofs

Consider the prolog goal state as a multiset


In [ ]:
class OrderedRes:
    clauses: set[set[Id]]
    rules: set[tuple[Id, set[Id]]]

In [ ]:
from dataclasses import dataclass, field
type Id = int
@dataclass
class OrderedRes:
    rules : dict[Id, set[Id]] = field(default_factory=dict)

    def makeprop(self):


    def reduce(self, c : set[Id]) -> set[Id]: # "find", "sld",  ordered_sld, query, query_remainder, query_quotient
        res = set()
        c = set(c)
        while c:
            #p = c.pop() # pop max. Use heap?
            #assert all(-q not in c for q in c)
            p = max(c, key=abs)
            c.remove(p)
            # What if p is negative? So what.
            body = self.rules.get(p)
            if body is None:
                res.add(p)
            else:
                c.update(body)
        return res
    
    def query(self, goals : set[Id]) -> set[Id]: # "find", "sld",  ordered_sld, query, query_remainder, query_quotient
        rem = set()
        goals = set(goals)
        while goals:
            #p = c.pop() # pop max. Use heap?
            #assert all(-q not in c for q in c)
            top_goal = max(goals, key=abs)
            c.remove(top_goal)
            # What if p is negative? So what.
            body = self.rules.get(top_goal)
            if body is None:
                # we could not prove the goal and can print a reaminder needed
                rem.add(p)
            else:
                goals.update(body)
        return rem

    def add_horn(self, head : Id, body : set[Id]):
        # horn is considered to just a convnience
        # If truly oriented, like refinement rewriting, something else is needed.
        clause = {-p for p in body}
        clause.add(head)
        self.add_clause(clause)
        
    def add_fact(self, p : Id):
        self.add_clause({p})

    def add_clause(self, c : set[Id]): # "union"
        p = max(c, key=abs) # maybe tie break by sign? (abs(p), p // abs(p))
        # if you ever have p and -p in a clause, you're inconsistant (unsat)
        assert -p not in c
        assert all(-q not in c for q in c)
        c.remove(p)
        c = {-x for x in c}
        c = self.reduce(c)
        body = self.rules.get(p)
        #negbody = self.rules.get(-p)
        if body is None:
            self.rules[p] = c
            return c
        else:
            body = self.reduce(body)
            if body == c:
                return body
            else:
                res = self.add_clause(c + body)
                self.rules[p] = res
                return res

    def compress(self):
        # this isn't rebuilding in the sense of egraphs or critical pairs
        # It is more like R-simplify rules
        for p, body in self.rules.items():
            self.rules[p] = self.reduce(body)

    def check_sat(self):
        # should contradiction be manifestly obvious?
        for p, body in self.rules.items():
            body2 = self.rules.get(-p)
            if body2 is not None:
                if len(self.reduce(body | body2)) == 0:
                    return False
        return True

R = OrderedRes()
#R.add_clause({3,-2,-1}) # clause 3 : -2, -1
#R.reduce({3})
R.add_horn(3, {1,2})
assert R.reduce({3}) == {1,2}
R.add_fact(1)
assert R.reduce({3}) == {2} # Now only 2 is necessary
R.add_horn(2, {1})
assert R.reduce({3}) == set() # nothing extra necessary
#R.add_clause({-2,3,1})
R.add_fact(-1) # hmm
R

# it's kind of a compressed database of conditional facts.
# Conditional datalog / hypothetical datalog

R.check_sat()


False

In [ ]:
@dataclass
class OrderedRes2:
    memo : dict[object, Id] = field(default_factory=dict)
    nodes : list[object] = field(default_factory=list)
    rules : OrderedRes = field(default_factory=OrderedRes)
    def prop(self, x) -> Id:
        if x not in self.memo:
            self.memo[x] = len(self.nodes) + 1
            self.nodes.append(x)
        return self.memo[x]
    def query(self, xs : object) -> Id:
        xs = {self.prop(x) for x in xs}
        return self.rules.reduce(xs)
    def add_horn(self, head : object, body : set[object]):
        head = self.prop(head)
        body = {self.prop(x) for x in body}
        self.rules.add_horn(head, body)
    def add_fact(self, x):
        x = self.prop(x)
        self.rules.add_fact(x)

In [17]:
from kdrag.all import *
import kdrag.property

T = smt.DeclareSort("T")
join = smt.Function("T.join", T, T, T)
kd.notation.or_.register(T, join)
x,y,z = smt.Consts("x y z", T)
join_semilattice = kd.define("T.join_semilattice", [], smt.And(
    kd.property.Assoc(join), kd.property.Comm(join), kd.property.Idem(join)
))

le = kd.notation.le.define([x,y], x | y == y)

zero = smt.Const("zero", T)
bounded = kd.define("T.bounded", [], 
    smt.ForAll([x,y], x | zero == x)
)

meet = smt.Function("T.meet", T, T, T)
kd.notation.and_.register(T, meet)
meet_semilattice = kd.define("T.meet_semilattice", [], smt.And(
    kd.property.Assoc(meet), kd.property.Comm(meet), kd.property.Idem(meet)
))

lattice = kd.define("T.lattice", [], smt.And(
    kd.property.Comm(join), kd.property.Comm(meet), kd.property.Assoc(join), kd.property.Assoc(meet),
                                             smt.ForAll([x,y], x & (x | y) == x),
    smt.ForAll([x,y], x | (x & y) == x)
    ))



one = smt.Const("one", T)
meet_bounded = kd.define("T.meet_bounded", [],
    smt.ForAll([x,y], x & one == x)
)



# the homomorphism theorems.
# heyting
impl = smt.Function("impl", T,T,T)
kd.notation.sub.register(T, lambda x,y: impl(y,x))
a,b,c = smt.Consts("a b c", T)
complement = kd.define("T.complement", [], (a <= c - b) == (a & b <= c))
#heyting = kd.define("T.heyting", [], smt.And(lattice, complement, bounded, meet_bounded))
heyting = kd.define("T.heyting", [], smt.And(lattice, 
    smt.ForAll([x], (x | zero) == x), smt.ForAll([x], (x & one) == x),
    smt.ForAll([a,b,c], (a <= impl(b,c)) == ((a & b) <= c)))
)

#CRUSH = [heyting.defn, complement.defn, join_semilattice.defn, meet_semilattice.defn, lattice.defn, le.defn, bounded.defn, meet_bounded.defn]
CRUSH = [lattice.defn, meet_semilattice.defn, join_semilattice.defn]
#kd.tactics.vprove(smt.ForAll([a,b,c], lattice, (a & (b | c)) == ((a & b) | (a & c))), by=CRUSH)
join_idem = kd.tactics.vprove(smt.Implies(lattice, kd.property.Idem(join)), by=[lattice.defn])
meet_idem = kd.tactics.vprove(smt.Implies(lattice, kd.property.Idem(meet)), by=[lattice.defn])
kd.tactics.prove(smt.Implies(heyting, impl(a,a) == one), by=[lattice.defn, heyting.defn, le.defn])
kd.tactics.vprove(smt.Implies(heyting, a & impl(a,b) == a & b), by=[lattice.defn, heyting.defn, le.defn])
kd.tactics.vprove(smt.Implies(heyting, impl(a,b) & b == b), by=[lattice.defn, heyting.defn, le.defn])
kd.tactics.vprove(smt.Implies(heyting, impl(a, b & c) == (impl(a, b) & impl(a, c))), by=[lattice.defn, heyting.defn, le.defn])
#kd.tactics.vprove(smt.ForAll([a], heyting, a - a == zero), by=CRUSH)


TimeoutError: Timeout. Maybe you have given `prove` too many or not enough lemmas?

In [9]:
le.defn

|= ForAll([x, y], T.le(x, y) == (T.join(x, y) == y))

In [2]:
meet_semilattice.defn

|= T.meet_semilattice ==
And(ForAll([x, y, z],
           T.meet(T.meet(x, y), z) ==
           T.meet(x, T.meet(y, z))),
    ForAll([x, y], T.meet(x, y) == T.meet(y, x)),
    ForAll(x, T.meet(x, x) == x))